In [2]:
import pandas as pd
from langchain_community.vectorstores import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import TextLoader
import re
# import io
import os
from tqdm import tqdm # Barra de progresso
import matplotlib.pyplot as plt # Para analise de resultados 
import seaborn as sns # Para analise de resultados
from concurrent.futures import ThreadPoolExecutor


import os
import re
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

# LangChain e Ollama Imports
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.chat_models import ChatOllama


In [3]:
import os
import re
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

# LangChain e Ollama Imports
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ========================================================================================
# --- 1. FERRAMENTAS BASE & REGEX ---
# ========================================================================================

CID10_RE = re.compile(r"\b([C-D]\d{2}(?:\.\d{1,2})?)\b", re.IGNORECASE)

def extract_cids(text: str) -> set[str]:
    """Extrai todos os CIDs válidos de um texto para checagem pós-LLM."""
    if not text:
        return set()
    return set(m.group(1).upper().replace(".", "") for m in CID10_RE.finditer(text))

# ========================================================================================
# --- 2. CLASSIFICADOR DETERMINÍSTICO (REGRAS EXATAS - SUBSTITUI A PARTE 1 ANTIGA) ---
# ========================================================================================

class ClassificadorOncologicoRegras:
    def __init__(self, crosswalk_file: str):
        self.df_crosswalk = pd.DataFrame()
        try:
            self.df_crosswalk = pd.read_csv(crosswalk_file, sep=';', dtype=str).fillna('')
            self.df_crosswalk['SNOMED_norm'] = self.df_crosswalk['SNOMED'].str.strip().str.upper()
            self.df_crosswalk['Topografia_norm'] = self.df_crosswalk['Topografia'].str.strip().str.lower()
        except Exception as e:
            print(f"Erro ao carregar matriz de regras: {e}")

    def avaliar_caso(self, diagnosis: str, topography: str, snomed_code: str):
        topo_lower = str(topography).lower().strip()
        snomed_str = str(snomed_code).strip().upper()
        
        if snomed_str and len(snomed_str) >= 5 and not snomed_str.startswith('M-'):
            snomed_str = f"M-{snomed_str}"
            
        resultado = {
            "CID_Sugerido": "INDETERMINADO",
            "Contexto_Regra_Para_LLM": ""
        }
        
        # 1. Tenta achar o CID Exato na tabela
        if not self.df_crosswalk.empty and snomed_str:
            matches = self.df_crosswalk[self.df_crosswalk['SNOMED_norm'] == snomed_str]
            for _, row in matches.iterrows():
                topo_csv = row['Topografia_norm']
                if (topo_csv in topo_lower) or (topo_lower in topo_csv) or (topo_csv == "nenhuma_topografia_encontrada"):
                    cid_encontrado = str(row['CID']).strip().upper()
                    if cid_encontrado:
                        resultado["CID_Sugerido"] = cid_encontrado
                        return resultado

        # 2. Se falhou na busca exata, extrai a DICA do Comportamento SNOMED para o LLM
        if snomed_str and '/' in snomed_str:
            last_digit = snomed_str.split('/')[-1]
            if last_digit == '0': resultado["Contexto_Regra_Para_LLM"] = "ATENÇÃO: SNOMED indica comportamento BENIGNO (/0). O CID deve pertencer preferencialmente ao bloco D10-D36."
            elif last_digit == '2': resultado["Contexto_Regra_Para_LLM"] = "ATENÇÃO: SNOMED indica In Situ (/2). CID deve pertencer ao bloco D00-D09."
            elif last_digit == '3': resultado["Contexto_Regra_Para_LLM"] = "ATENÇÃO: SNOMED indica Maligno (/3). CID deve pertencer ao bloco C00-C97."
            elif last_digit == '6': resultado["Contexto_Regra_Para_LLM"] = "ATENÇÃO: SNOMED indica Metástase (/6). Avaliar CIDs de neoplasias secundárias (Ex: C77-C79)."
            elif last_digit == '1': resultado["Contexto_Regra_Para_LLM"] = "ATENÇÃO: SNOMED indica comportamento incerto (/1). CID deve pertencer ao bloco D37-D48."

        return resultado

# ========================================================================================
# --- 3. CARREGAMENTO DOS VETORES (PARA O RAG) ---
# ========================================================================================

def load_snomed_csv_as_documents(filepath):
    """Carrega o CSV de conversão e foca apenas em ensinar SNOMED+Topografia = CID ao LangChain."""
    documents_list = []
    try:
        df = pd.read_csv(filepath, sep=';').fillna('')
        for index, row in df.iterrows():
            snomed = str(row.get('SNOMED', '')).strip()
            diag = str(row.get('Diagnóstico', '')).strip()
            topo = str(row.get('Topografia', '')).strip()
            cid = str(row.get('CID', '')).strip()
            desc_cid = str(row.get('Descrição CID', '')).strip()
            
            # Só indexa linhas que de fato têm um CID para ensinar
            if not cid: continue 

            # Texto altamente otimizado para a busca vetorial (Embedding)
            content = f"SNOMED: {snomed}. Diagnóstico: {diag}. Topografia: {topo}. CID Correspondente: {cid} ({desc_cid})."
            
            doc = Document(
                page_content=content,
                metadata={"source": filepath, "cid_target": cid}
            )
            documents_list.append(doc)
            
        print(f"RAG: {len(documents_list)} documentos carregados de {filepath}.")
    except Exception as e:
        print(f"Erro ao converter CSV para docs LangChain: {e}")
        
    return documents_list

print("1. Iniciando carregamento dos documentos para o RAG...")
docs_csv = load_snomed_csv_as_documents("SNOMED_to_CID_2.csv")

print("2. Inicializando ChromaDB local...")
embeddings = OllamaEmbeddings(model="nomic-embed-text") 
persist_dir = "./chroma_db_local"

if os.path.exists(persist_dir) and os.listdir(persist_dir):
    vectorstore = Chroma(persist_directory=persist_dir, embedding_function=embeddings)
else:
    vectorstore = Chroma.from_documents(docs_csv, embeddings, persist_directory=persist_dir)
    
retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 5, "fetch_k": 15})

# --- LLM LOCAL ---
llm = ChatOllama(model="gemma3:12b", temperature=0, num_ctx=4096)

# ========================================================================================
# --- 4. PROMPT E RAG CHAIN ---
# ========================================================================================

PROMPT_SISTEMA = """
# PERSONA
Você é um auditor médico especialista em codificação oncológica (CID-O-3 e CID-10).

# OBJETIVO
Sua única tarefa é analisar os dados do caso e encontrar o código CID exato correspondente no CONTEXTO.

# REGRAS
1) Responda APENAS com o código do CID (Exemplo: C349).
2) Se houver múltiplos, separe por ponto e vírgula (Exemplo: C56;C57).
3) Leia atentamente a <DICA_DA_REGRA_SISTEMICA>. Se ela informar que o comportamento é benigno ou maligno, garanta que o CID escolhido do contexto respeite essa diretriz (Ex: C é maligno, D10-D36 é benigno).
4) Se nenhum CID no contexto for aplicável para a combinação informada, responda: Nenhum_CID_encontrado.
5) Não inclua textos explicativos, nem cabeçalhos.
"""

PROMPT_USUARIO = """
<CONTEXTO_RECUPERADO_DA_BASE_DE_DADOS>
{contexto}
</CONTEXTO_RECUPERADO_DA_BASE_DE_DADOS>

<DADOS_DO_CASO_PARA_ANALISE>
  <SNOMED>{snomed}</SNOMED>
  <DIAGNOSTICO>{label}</DIAGNOSTICO>
  <TOPOGRAFIA>{topografia}</TOPOGRAFIA>
  <PROCEDIMENTO>{procedimento}</PROCEDIMENTO>
</DADOS_DO_CASO_PARA_ANALISE>

<DICA_DA_REGRA_SISTEMICA>
{dica_regra}
</DICA_DA_REGRA_SISTEMICA>

Qual o CID?
"""

prompt_template = ChatPromptTemplate.from_messages([("system", PROMPT_SISTEMA), ("human", PROMPT_USUARIO)])
rag_chain = prompt_template | llm | StrOutputParser()

# ========================================================================================
# --- 5. EXECUÇÃO DA PIPELINE HÍBRIDA ---
# ========================================================================================

classificador_regras = ClassificadorOncologicoRegras("SNOMED_to_CID_2.csv")

print("3. Lendo arquivo de dados a serem processados...")
# Supondo que você está lendo dados brutos para classificar 
# (Se for o mesmo SNOMED_to_CID_2, servirá como teste prático da pipeline)
df_original = pd.read_csv('Resultado_SNOMED_CID_Parcial - Copia.csv', sep=';').fillna('Não informado')

if 'DesProcedimento' not in df_original.columns:
    df_original['DesProcedimento'] = "Não informado"

# Deduplicação baseada em campos chave
df_original['chave_unica'] = (
    df_original['Diagnóstico'].astype(str) + "|" + 
    df_original['Topografia'].astype(str) + "|" + 
    df_original['SNOMED'].astype(str)
)

df_unicos = df_original.drop_duplicates(subset=['chave_unica']).copy()
print(f"-> Total de casos únicos para processar: {len(df_unicos)}")

nome_checkpoint = 'checkpoint_classificacao.csv'
casos_ja_feitos = set()
resultados_processados = []

if os.path.exists(nome_checkpoint):
    try:
        df_check = pd.read_csv(nome_checkpoint, sep=';')
        resultados_processados = df_check.to_dict('records')
        casos_ja_feitos = set(df_check['chave_unica'].values)
        print(f"-> Retomando: {len(casos_ja_feitos)} processados no checkpoint.")
    except: pass

def process_row(row_dict):
    chave = row_dict['chave_unica']
    if chave in casos_ja_feitos: return None

    diag = str(row_dict.get('Diagnóstico', ''))
    topo = str(row_dict.get('Topografia', ''))
    snomed = str(row_dict.get('SNOMED', ''))
    proc = str(row_dict.get('DesProcedimento', ''))

    # PASSO A: Tenta o motor de regras
    resultado_regra = classificador_regras.avaliar_caso(diag, topo, snomed)
    
    cid_final = resultado_regra["CID_Sugerido"]
    metodo = "REGRA_EXATA"

    # PASSO B: RAG (Se a regra não achar um cruzamento exato)
    if cid_final == "INDETERMINADO":
        metodo = "RAG_LLM"
        topo_llm = "" if topo in ["Nenhuma_topografia_encontrada", "Muitas_topografias_possiveis"] else topo
        
        try:
            # Query focada nas entidades chaves
            query = f"SNOMED: {snomed} Diagnóstico: {diag} Topografia: {topo_llm}"
            docs_recuperados = retriever.invoke(query)
            context_text = "\n".join([d.page_content for d in docs_recuperados])

            response = rag_chain.invoke({
                "contexto": context_text,
                "snomed": snomed,
                "label": diag,
                "topografia": topo_llm,
                "procedimento": proc,
                "dica_regra": resultado_regra["Contexto_Regra_Para_LLM"] # Passa a dica do Python para o LLM
            })

            cid_final = response.strip().split('\n')[0].replace(".", "").replace('"', '').strip()

            # Filtro de alucinação (Valida se o LLM não tirou um CID da cartola)
            cids_no_contexto = extract_cids(context_text)
            if cid_final != "Nenhum_CID_encontrado":
                predicted = [c.strip() for c in cid_final.split(";") if c.strip() in cids_no_contexto]
                cid_final = ";".join(predicted) if predicted else "Nenhum_CID_encontrado"

        except Exception as e:
            cid_final = "ERRO"
            metodo = "ERRO_RAG"
            print(f"Erro IA no caso {chave}: {e}")

    return {
        "chave_unica": chave,
        "CID_Final": cid_final,
        "Metodo": metodo
    }

print("4. Iniciando processamento híbrido...")

rows = df_unicos.to_dict("records")
batch_save = 20

with ThreadPoolExecutor(max_workers=3) as executor:
    for resultado in tqdm(executor.map(process_row, rows), total=len(rows)):
        if resultado:
            resultados_processados.append(resultado)
            if len(resultados_processados) % batch_save == 0:
                pd.DataFrame(resultados_processados).to_csv(nome_checkpoint, sep=';', index=False, encoding='utf-8-sig')

# Salva final
pd.DataFrame(resultados_processados).to_csv("classificacao_final.csv", sep=';', index=False, encoding='utf-8-sig')
print("Processamento concluído com sucesso!")

1. Iniciando carregamento dos documentos para o RAG...


C:\Users\rafae\AppData\Local\Temp\ipykernel_2756\2593636764.py:116: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=persist_dir, embedding_function=embeddings)
C:\Users\rafae\AppData\Local\Temp\ipykernel_2756\2593636764.py:123: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(model="gemma3:12b", temperature=0, num_ctx=4096)


RAG: 7304 documentos carregados de SNOMED_to_CID_2.csv.
2. Inicializando ChromaDB local...
3. Lendo arquivo de dados a serem processados...
-> Total de casos únicos para processar: 55
4. Iniciando processamento híbrido...


100%|██████████| 55/55 [01:25<00:00,  1.55s/it]

Processamento concluído com sucesso!


In [4]:
# ==============================================================================
# 5. MERGE FINAL E SALVAMENTO
# ==============================================================================
print("5. Gerando arquivo final completo...")

df_final = pd.merge(df_original, df_resultados_unicos, on='chave_unica', how='left')

cols = ['SNOMED', 'Diagnóstico', 'Topografia', 'DesProcedimento', 'CID_Preditos', 'Metodo_Utilizado', 'Status_Regra']
cols += [c for c in df_final.columns if c not in cols and c != 'chave_unica']
df_final = df_final[cols]

df_final.to_csv("CID_paciente_CLASSIFICADO_FINAL.csv", sep=';', index=False, encoding='utf-8-sig')
print("SUCESSO! Arquivo 'CID_paciente_CLASSIFICADO_FINAL.csv' gerado.")


5. Gerando arquivo final completo...
SUCESSO! Arquivo 'CID_paciente_CLASSIFICADO_FINAL.csv' gerado.


In [ ]:
# ==============================================================================
# 6. ANÁLISE DE RESULTADOS
# ==============================================================================


plt.figure(figsize=(10, 6))
counts_metodo = df_final['Metodo_Utilizado'].value_counts()

# Gráfico de Pizza
plt.pie(counts_metodo, labels=counts_metodo.index, autopct='%1.1f%%', startangle=140, colors=['#66b3ff','#99ff99','#ffcc99'])
plt.title('Eficiência da Triagem: Regras Determinísticas vs Inteligência Artificial')
# plt.savefig("relatorios_img/kpi_eficiencia_automacao.png")
# plt.close()

print(f"-> Gráfico de Eficiência salvo em relatorios_img")

NameError: name 'df_final' is not defined

<Figure size 1000x600 with 0 Axes>

In [ ]:
import os
import re
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

# LangChain e Ollama Imports
from langchain.schema import Document
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ========================================================================================
# --- 1. FERRAMENTAS BASE & REGEX ---
# ========================================================================================

CID10_RE = re.compile(r"\b([C-D]\d{2}(?:\.\d{1,2})?)\b", re.IGNORECASE)

def extract_cids(text: str) -> set[str]:
    """Extrai todos os CIDs válidos de um texto para checagem pós-LLM."""
    if not text:
        return set()
    return set(m.group(1).upper().replace(".", "") for m in CID10_RE.finditer(text))

# ========================================================================================
# --- 2. CLASSIFICADOR DETERMINÍSTICO (REGRAS EXATAS - SUBSTITUI A PARTE 1 ANTIGA) ---
# ========================================================================================

class ClassificadorOncologicoRegras:
    def __init__(self, crosswalk_file: str):
        self.df_crosswalk = pd.DataFrame()
        try:
            self.df_crosswalk = pd.read_csv(crosswalk_file, sep=';', dtype=str).fillna('')
            self.df_crosswalk['SNOMED_norm'] = self.df_crosswalk['SNOMED'].str.strip().str.upper()
            self.df_crosswalk['Topografia_norm'] = self.df_crosswalk['Topografia'].str.strip().str.lower()
        except Exception as e:
            print(f"Erro ao carregar matriz de regras: {e}")

    def avaliar_caso(self, diagnosis: str, topography: str, snomed_code: str):
        topo_lower = str(topography).lower().strip()
        snomed_str = str(snomed_code).strip().upper()
        
        if snomed_str and len(snomed_str) >= 5 and not snomed_str.startswith('M-'):
            snomed_str = f"M-{snomed_str}"
            
        resultado = {
            "CID_Sugerido": "INDETERMINADO",
            "Contexto_Regra_Para_LLM": ""
        }
        
        # 1. Tenta achar o CID Exato na tabela
        if not self.df_crosswalk.empty and snomed_str:
            matches = self.df_crosswalk[self.df_crosswalk['SNOMED_norm'] == snomed_str]
            for _, row in matches.iterrows():
                topo_csv = row['Topografia_norm']
                if (topo_csv in topo_lower) or (topo_lower in topo_csv) or (topo_csv == "nenhuma_topografia_encontrada"):
                    cid_encontrado = str(row['CID']).strip().upper()
                    if cid_encontrado:
                        resultado["CID_Sugerido"] = cid_encontrado
                        return resultado

        # 2. Se falhou na busca exata, extrai a DICA do Comportamento SNOMED para o LLM
        if snomed_str and '/' in snomed_str:
            last_digit = snomed_str.split('/')[-1]
            if last_digit == '0': resultado["Contexto_Regra_Para_LLM"] = "ATENÇÃO: SNOMED indica comportamento BENIGNO (/0). O CID deve pertencer preferencialmente ao bloco D10-D36."
            elif last_digit == '2': resultado["Contexto_Regra_Para_LLM"] = "ATENÇÃO: SNOMED indica In Situ (/2). CID deve pertencer ao bloco D00-D09."
            elif last_digit == '3': resultado["Contexto_Regra_Para_LLM"] = "ATENÇÃO: SNOMED indica Maligno (/3). CID deve pertencer ao bloco C00-C97."
            elif last_digit == '6': resultado["Contexto_Regra_Para_LLM"] = "ATENÇÃO: SNOMED indica Metástase (/6). Avaliar CIDs de neoplasias secundárias (Ex: C77-C79)."
            elif last_digit == '1': resultado["Contexto_Regra_Para_LLM"] = "ATENÇÃO: SNOMED indica comportamento incerto (/1). CID deve pertencer ao bloco D37-D48."

        return resultado

# ========================================================================================
# --- 3. CARREGAMENTO DOS VETORES (PARA O RAG) ---
# ========================================================================================

def load_snomed_csv_as_documents(filepath):
    """Carrega o CSV de conversão e foca apenas em ensinar SNOMED+Topografia = CID ao LangChain."""
    documents_list = []
    try:
        df = pd.read_csv(filepath, sep=';').fillna('')
        for index, row in df.iterrows():
            snomed = str(row.get('SNOMED', '')).strip()
            diag = str(row.get('Diagnóstico', '')).strip()
            topo = str(row.get('Topografia', '')).strip()
            cid = str(row.get('CID', '')).strip()
            desc_cid = str(row.get('Descrição CID', '')).strip()
            
            # Só indexa linhas que de fato têm um CID para ensinar
            if not cid: continue 

            # Texto altamente otimizado para a busca vetorial (Embedding)
            content = f"SNOMED: {snomed}. Diagnóstico: {diag}. Topografia: {topo}. CID Correspondente: {cid} ({desc_cid})."
            
            doc = Document(
                page_content=content,
                metadata={"source": filepath, "cid_target": cid}
            )
            documents_list.append(doc)
            
        print(f"RAG: {len(documents_list)} documentos carregados de {filepath}.")
    except Exception as e:
        print(f"Erro ao converter CSV para docs LangChain: {e}")
        
    return documents_list

print("1. Iniciando carregamento dos documentos para o RAG...")
docs_csv = load_snomed_csv_as_documents("SNOMED_to_CID_2.csv")

print("2. Inicializando ChromaDB local...")
embeddings = OllamaEmbeddings(model="nomic-embed-text") 
persist_dir = "./chroma_db_local"

if os.path.exists(persist_dir) and os.listdir(persist_dir):
    vectorstore = Chroma(persist_directory=persist_dir, embedding_function=embeddings)
else:
    vectorstore = Chroma.from_documents(docs_csv, embeddings, persist_directory=persist_dir)
    
retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 5, "fetch_k": 15})

# --- LLM LOCAL ---
llm = ChatOllama(model="gemma3:12b", temperature=0, num_ctx=4096)

# ========================================================================================
# --- 4. PROMPT E RAG CHAIN ---
# ========================================================================================

PROMPT_SISTEMA = """
# PERSONA
Você é um auditor médico especialista em codificação oncológica (CID-O-3 e CID-10).

# OBJETIVO
Sua única tarefa é analisar os dados do caso e encontrar o código CID exato correspondente no CONTEXTO.

# REGRAS
1) Responda APENAS com o código do CID (Exemplo: C349).
2) Se houver múltiplos, separe por ponto e vírgula (Exemplo: C56;C57).
3) Leia atentamente a <DICA_DA_REGRA_SISTEMICA>. Se ela informar que o comportamento é benigno ou maligno, garanta que o CID escolhido do contexto respeite essa diretriz (Ex: C é maligno, D10-D36 é benigno).
4) Se nenhum CID no contexto for aplicável para a combinação informada, responda: Nenhum_CID_encontrado.
5) Não inclua textos explicativos, nem cabeçalhos.
"""

PROMPT_USUARIO = """
<CONTEXTO_RECUPERADO_DA_BASE_DE_DADOS>
{contexto}
</CONTEXTO_RECUPERADO_DA_BASE_DE_DADOS>

<DADOS_DO_CASO_PARA_ANALISE>
  <SNOMED>{snomed}</SNOMED>
  <DIAGNOSTICO>{label}</DIAGNOSTICO>
  <TOPOGRAFIA>{topografia}</TOPOGRAFIA>
  <PROCEDIMENTO>{procedimento}</PROCEDIMENTO>
</DADOS_DO_CASO_PARA_ANALISE>

<DICA_DA_REGRA_SISTEMICA>
{dica_regra}
</DICA_DA_REGRA_SISTEMICA>

Qual o CID?
"""

prompt_template = ChatPromptTemplate.from_messages([("system", PROMPT_SISTEMA), ("human", PROMPT_USUARIO)])
rag_chain = prompt_template | llm | StrOutputParser()

# ========================================================================================
# --- 5. EXECUÇÃO DA PIPELINE HÍBRIDA ---
# ========================================================================================

classificador_regras = ClassificadorOncologicoRegras("SNOMED_to_CID_2.csv")

print("3. Lendo arquivo de dados a serem processados...")
# Supondo que você está lendo dados brutos para classificar 
# (Se for o mesmo SNOMED_to_CID_2, servirá como teste prático da pipeline)
df_original = pd.read_csv('Resultado_SNOMED_CID_Parcial - Copia.csv', sep=';').fillna('Não informado')

if 'DesProcedimento' not in df_original.columns:
    df_original['DesProcedimento'] = "Não informado"

# Deduplicação baseada em campos chave
df_original['chave_unica'] = (
    df_original['Diagnóstico'].astype(str) + "|" + 
    df_original['Topografia'].astype(str) + "|" + 
    df_original['SNOMED'].astype(str)
)

df_unicos = df_original.drop_duplicates(subset=['chave_unica']).copy()
print(f"-> Total de casos únicos para processar: {len(df_unicos)}")

nome_checkpoint = 'checkpoint_classificacao.csv'
casos_ja_feitos = set()
resultados_processados = []

if os.path.exists(nome_checkpoint):
    try:
        df_check = pd.read_csv(nome_checkpoint, sep=';')
        resultados_processados = df_check.to_dict('records')
        casos_ja_feitos = set(df_check['chave_unica'].values)
        print(f"-> Retomando: {len(casos_ja_feitos)} processados no checkpoint.")
    except: pass

def process_row(row_dict):
    chave = row_dict['chave_unica']
    if chave in casos_ja_feitos: return None

    diag = str(row_dict.get('Diagnóstico', ''))
    topo = str(row_dict.get('Topografia', ''))
    snomed = str(row_dict.get('SNOMED', ''))
    proc = str(row_dict.get('DesProcedimento', ''))

    # PASSO A: Tenta o motor de regras
    resultado_regra = classificador_regras.avaliar_caso(diag, topo, snomed)
    
    cid_final = resultado_regra["CID_Sugerido"]
    metodo = "REGRA_EXATA"

    # PASSO B: RAG (Se a regra não achar um cruzamento exato)
    if cid_final == "INDETERMINADO":
        metodo = "RAG_LLM"
        topo_llm = "" if topo in ["Nenhuma_topografia_encontrada", "Muitas_topografias_possiveis"] else topo
        
        try:
            # Query focada nas entidades chaves
            query = f"SNOMED: {snomed} Diagnóstico: {diag} Topografia: {topo_llm}"
            docs_recuperados = retriever.invoke(query)
            context_text = "\n".join([d.page_content for d in docs_recuperados])

            response = rag_chain.invoke({
                "contexto": context_text,
                "snomed": snomed,
                "label": diag,
                "topografia": topo_llm,
                "procedimento": proc,
                "dica_regra": resultado_regra["Contexto_Regra_Para_LLM"] # Passa a dica do Python para o LLM
            })

            cid_final = response.strip().split('\n')[0].replace(".", "").replace('"', '').strip()

            # Filtro de alucinação (Valida se o LLM não tirou um CID da cartola)
            cids_no_contexto = extract_cids(context_text)
            if cid_final != "Nenhum_CID_encontrado":
                predicted = [c.strip() for c in cid_final.split(";") if c.strip() in cids_no_contexto]
                cid_final = ";".join(predicted) if predicted else "Nenhum_CID_encontrado"

        except Exception as e:
            cid_final = "ERRO"
            metodo = "ERRO_RAG"
            print(f"Erro IA no caso {chave}: {e}")

    return {
        "chave_unica": chave,
        "CID_Final": cid_final,
        "Metodo": metodo
    }

print("4. Iniciando processamento híbrido...")

rows = df_unicos.to_dict("records")
batch_save = 20

with ThreadPoolExecutor(max_workers=3) as executor:
    for resultado in tqdm(executor.map(process_row, rows), total=len(rows)):
        if resultado:
            resultados_processados.append(resultado)
            if len(resultados_processados) % batch_save == 0:
                pd.DataFrame(resultados_processados).to_csv(nome_checkpoint, sep=';', index=False, encoding='utf-8-sig')

# Salva final
pd.DataFrame(resultados_processados).to_csv("classificacao_final.csv", sep=';', index=False, encoding='utf-8-sig')
print("Processamento concluído com sucesso!")